# Phase 3 — PointNet++ on S3DIS (Colab GPU)

Trains PointNet++ on Areas 1-4,6 and predicts **Area-5 at full resolution**, exporting per-point predictions that `scripts/eval_pointnet2_s3dis.py` scores **locally** with the shared global-mIoU evaluator — apples-to-apples with the geometry / feature_ml / hybrid arms.

**Before running, do this once on your machine:**
```bash
python scripts/pack_for_colab.py            # -> data/s3dis/colab_pack/Area_*.npz  (~4 GB)
```
Then upload the 6 `Area_*.npz` files to Google Drive under a folder named **`s3dis_pack`** (`MyDrive/s3dis_pack/Area_1.npz` ...).

**Runtime → Change runtime type → GPU (T4)** before running the cells below.

In [ ]:
# 1. Confirm a GPU is attached
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > GPU')

In [ ]:
# 2. Get the training code (this repo)
!git clone --depth 1 https://github.com/DsThakurRawat/Geometric-Scene-Architect.git repo
%cd repo

In [ ]:
# 3. Mount Google Drive (holds the uploaded s3dis_pack/ and receives the predictions)
from google.colab import drive
drive.mount('/content/drive')
import os
DATA_DIR = '/content/drive/MyDrive/s3dis_pack'
assert os.path.isdir(DATA_DIR), f'Upload the packed Area_*.npz to {DATA_DIR} first'
print('packs:', sorted(os.listdir(DATA_DIR)))

In [ ]:
# 4. Train on Areas 1-4,6 and predict Area-5 (a few hours on a T4).
#    Predictions + checkpoint go to Drive so they survive a disconnect.
!python colab/pointnet2_train.py \
    --data-dir "$DATA_DIR" \
    --epochs 32 --batch-size 16 \
    --out /content/drive/MyDrive/pointnet2_area5_preds.npz \
    --ckpt /content/drive/MyDrive/pointnet2_s3dis.pth

## 5. Score locally (on your machine)

Download `pointnet2_area5_preds.npz` from Drive into `outputs/`, then:
```bash
python scripts/eval_pointnet2_s3dis.py --preds outputs/pointnet2_area5_preds.npz
python scripts/make_figures.py --figdir docs/images   # adds PN++ to the per-class figure
```
This writes `outputs/pointnet2_eval.json` (global mIoU, FULL13) — the PointNet++ row for the multi-arm comparison.